In [1]:
import numpy as np
import  pandas as pd
import plotly.express as px

In [2]:
growth_rate = pd.read_csv("../results/growth_rate.csv")

In [3]:
growth_rate.head()

,species,y,x,mass,age,fitness,growth_rate,N_comp,P_comp,NP_ratio
0,1,120,3,0.026976,11.0,0.967651,0.008188,0.016945,0.007438,2.278051
1,1,23,91,0.051832,2.0,0.968068,0.015788,0.016879,0.008023,2.103718
2,1,55,124,0.047207,22.0,0.968888,0.014431,0.016755,0.008905,1.881580
3,1,59,39,0.039771,6.0,0.967384,0.011707,0.016981,0.007214,2.353913
4,1,112,49,0.045347,14.0,0.967607,0.013454,0.016949,0.007468,2.269368


In [4]:
fig = px.scatter(growth_rate, x="age", y="growth_rate", color="species", title="Growth Rate vs. Age")
fig.show()

In [5]:
import plotly.express as px

# Calculate mean and standard deviation for each age and species
growth_summary = (
    growth_rate
    .groupby(["age", "species"], as_index=False)
    .agg(
        mean_growth=("growth_rate", "mean"),
        sd_growth=("growth_rate", "std")
    )
)

# Plot mean growth rate with error bars (± 1 SD)
fig = px.scatter(
    growth_summary,
    x="age",
    y="mean_growth",
    color="species",
    error_y="sd_growth",
    title="Mean Growth Rate vs. Age",
    labels={
        "age": "Age",
        "mean_growth": "Mean Growth Rate",
        "sd_growth": "Standard Deviation"
    }
)

fig.show()

In [6]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy.optimize import curve_fit

# Your existing summary
growth_summary = (
    growth_rate
    .groupby(["age", "species"], as_index=False)
    .agg(
        mean_growth=("growth_rate", "mean"),
        sd_growth=("growth_rate", "std")
    )
)

# Gaussian growth-rate curve
def growth_curve(age, a, mu, sigma, c):
    return c + a * np.exp(-0.5 * ((age - mu) / sigma) ** 2)

fig = px.scatter(
    growth_summary,
    x="age",
    y="mean_growth",
    color="species",
    error_y="sd_growth",
    title="Mean Growth Rate vs. Age",
    labels={
        "age": "Age",
        "mean_growth": "Mean Growth Rate",
    }
)

# Fit and plot a separate curve for each species
for species, data in growth_summary.groupby("species"):
    data = data.dropna(subset=["age", "mean_growth"]).sort_values("age")

    if len(data) < 4 or data["age"].nunique() < 4:
        print(f"Skipping {species}: insufficient age points for Gaussian fit.")
        continue

    x = data["age"].to_numpy(dtype=float)
    y = data["mean_growth"].to_numpy(dtype=float)

    p0 = [
        max(y.max() - y.min(), 1e-6),
        x[np.argmax(y)],
        max(np.std(x), 1e-6),
        y.min()
    ]

    try:
        params, _ = curve_fit(
            growth_curve,
            x,
            y,
            p0=p0,
            bounds=(
                [0, x.min(), 1e-6, -np.inf],
                [np.inf, x.max(), np.inf, np.inf]
            ),
            maxfev=20000
        )

        x_fit = np.linspace(x.min(), x.max(), 300)
        y_fit = growth_curve(x_fit, *params)

        fig.add_trace(go.Scatter(
            x=x_fit,
            y=y_fit,
            mode="lines",
            name=f"{species} fit",
            legendgroup=str(species),
            line=dict(dash="solid", width=2)
        ))

        print(
            f"{species}: peak age = {params[1]:.2f}, "
            f"peak growth rate = {params[0] + params[3]:.3f}"
        )

    except RuntimeError:
        print(f"Fit failed for {species}")

fig.update_layout(legend_title_text="Species")
fig.show()

0: peak age = 0.00, peak growth rate = 0.013
1: peak age = 0.00, peak growth rate = 0.014
2: peak age = 0.00, peak growth rate = 0.013
3: peak age = 1.00, peak growth rate = 0.012
4: peak age = 0.00, peak growth rate = 0.014


In [ ]:
growth_summary[
    growth_summary["sd_growth"].isna()
]growth_s